# 818번 버스 도착 시간 분석

이 노트북은 818번 버스의 실제 운행 기록과 공식 시간표를 비교하여, 각 정류장별 도착 시간의 지연/조착을 분석합니다.

**분석 과정:**
1.  **환경 설정 및 데이터 로드**: 분석에 필요한 라이브러리를 설치하고, 전처리된 버스 운행 기록 및 관련 데이터를 불러옵니다.
2.  **데이터 필터링 및 준비**: 818번 버스 데이터만 필터링하고, 시간표 데이터를 분석하기 용이한 형태로 변환합니다.
3.  **시간표-운행 기록 매칭**: 모든 운행 날짜에 대해 시간표의 예정 시간과 가장 근접한 실제 도착 기록을 찾습니다.
4.  **결과 분석 및 저장**: 각 운행별 지연/조착 시간을 계산하고, 최종 분석 결과를 CSV 파일로 저장합니다.

## 1. 환경 설정 및 데이터 로드

In [1]:
# 데이터 분석 및 시각화에 필요한 라이브러리를 설치합니다.
!pip3 install pandas matplotlib seaborn

import pandas as pd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 한글 폰트 설정
font_path = '../_font/NanumGothic-Bold.ttf'
fm.fontManager.addfont(font_path)
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

print(f"Matplotlib 폰트가 '{font_prop.get_name()}'으로 설정되었습니다.")


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Matplotlib 폰트가 'NanumGothic'으로 설정되었습니다.
Matplotlib 폰트가 'NanumGothic'으로 설정되었습니다.


In [2]:
# 필요한 데이터들을 불러옵니다.
# 1. 전처리된 버스 이벤트 데이터
_gs_busevent_preprocessed = pd.read_csv("./data/_gs_busevent_preprocessed.csv")
# 2. 818번 버스 시간표
_bs_node_time = pd.read_csv("./data/818번 버스 데이터 통계 분석 - [데이터 기준] - 버스 시간표.csv")
# 3. 818번 버스 노선 정보
_bus_root = pd.read_csv("./data/bus_root.csv")

print("데이터 로드 완료.")
_gs_busevent_preprocessed.head()

데이터 로드 완료.


,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos,bstop_arrive_year,bstop_arrive_month,bstop_arrive_day,...,bstop_arrive_second,bstop_arrive_timezone,bstop_leave_year,bstop_leave_month,bstop_leave_day,bstop_leave_days,bstop_leave_hour,bstop_leave_minute,bstop_leave_second,bstop_leave_timezone
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229,2020,1,4,...,53,UTC+9,2020,1,4,2020-01-04,0,1,40,UTC+9
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398,2020,1,4,...,14,UTC+9,2020,1,4,2020-01-04,0,2,40,UTC+9
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077,2020,1,4,...,1,UTC+9,2020,1,4,2020-01-04,0,5,25,UTC+9
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677,2020,1,4,...,54,UTC+9,2020,1,4,2020-01-04,0,6,20,UTC+9
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354,2020,1,4,...,48,UTC+9,2020,1,4,2020-01-04,0,6,58,UTC+9


## 2. 데이터 필터링 및 준비

In [ ]:
# 818번 버스의 새벽 5시 이후 운행 기록만 필터링합니다.
# 2020년 1월 1일 이전 데이터는 분석에서 제외합니다.
df_bus_818 = _gs_busevent_preprocessed[
    (_gs_busevent_preprocessed['route_name'] == '818') &
    (_gs_busevent_preprocessed['bstop_arrive_time'] > 20200101235959) &
    (_gs_busevent_preprocessed['bstop_arrive_hour'] >= 5)
].sort_values(by='bstop_arrive_time', ascending=False)

# 시간표 데이터를 분석하기 용이한 'long format'으로 변환합니다.
df_schedule_long = _bs_node_time.melt(
    id_vars=['연번', '운행 경로 및 시간'],
    var_name='정류장',
    value_name='예정시간'
).dropna(subset=['예정시간'])

# 정류장 이름의 일관성을 위해 '.1'과 같은 불필요한 문자를 제거합니다.
df_schedule_long['정류장'] = df_schedule_long['정류장'].str.split('.').str[0]
df_schedule_long = df_schedule_long.sort_values(by=['연번', '예정시간'])

print("818번 버스 데이터 필터링 및 시간표 변환 완료.")
df_schedule_long.head()

818번 버스 데이터 필터링 및 시간표 변환 완료.


,연번,운행 경로 및 시간,정류장,예정시간
61,1,(하양대구은행 출발),대구대정문 건너,05:30:00
122,1,(하양대구은행 출발),하양역 건너,06:25:00
183,1,(하양대구은행 출발),상공회의소건너,07:05:00
244,1,(하양대구은행 출발),안심역(4번출구),07:15:00
305,1,(하양대구은행 출발),하양역,07:25:00


## 3. 시간표-운행 기록 매칭 및 지연/조착 분석

In [4]:
from tqdm import tqdm

# 모든 운행 날짜에 대해 분석을 수행합니다.
unique_dates = df_bus_818['bstop_arrive_days'].unique()
all_results = []

print(f"분석 대상 날짜: {unique_dates}")

for target_date in tqdm(unique_dates, desc="전체 날짜 분석 중"):
    # 해당 날짜의 운행 기록만 선택합니다.
    df_events_oneday = df_bus_818[df_bus_818['bstop_arrive_days'] == target_date].copy()
    if df_events_oneday.empty:
        continue

    # 시간 비교를 위해 datetime 객체로 변환합니다.
    df_events_oneday['arrive_time_dt'] = pd.to_datetime(df_events_oneday['bstop_arrive_time'], format='%Y%m%d%H%M%S')
    
    df_schedule_long_copy = df_schedule_long.copy()
    df_schedule_long_copy['예정시간_dt'] = pd.to_datetime(target_date + ' ' + df_schedule_long_copy['예정시간'], errors='coerce')
    df_schedule_long_oneday = df_schedule_long_copy.dropna(subset=['예정시간_dt'])

    # '정류장' 이름을 기준으로 시간표와 운행 기록을 병합합니다.
    merged_df = pd.merge(
        df_schedule_long_oneday,
        df_events_oneday.rename(columns={'node_name': '정류장'}),
        on='정류장',
        how='left'
    )

    # 예정 시간과 실제 도착 시간의 차이를 계산하여 가장 근접한 기록을 찾습니다.
    merged_df['time_diff'] = (merged_df['예정시간_dt'] - merged_df['arrive_time_dt']).dt.total_seconds().abs()
    merged_df.dropna(subset=['time_diff'], inplace=True)
    if merged_df.empty:
        continue
    
    idx = merged_df.groupby(['연번', '정류장', '예정시간_dt'])['time_diff'].idxmin()
    result_df_daily = merged_df.loc[idx].copy()

    # 지연/조착 시간을 분 단위로 계산합니다.
    result_df_daily['지연(분)'] = ((result_df_daily['arrive_time_dt'] - result_df_daily['예정시간_dt']).dt.total_seconds() / 60).round(2)
    
    # 결과 컬럼을 정리하고 리스트에 추가합니다.
    result_df_daily = result_df_daily[['연번', '정류장', '예정시간_dt', 'arrive_time_dt', '지연(분)']]
    result_df_daily.rename(columns={'예정시간_dt': '예정시간', 'arrive_time_dt': '실제도착시간'}, inplace=True)
    all_results.append(result_df_daily)

# 최종 결과를 하나의 데이터프레임으로 합치고 정렬합니다.
if all_results:
    final_result_df = pd.concat(all_results, ignore_index=True).sort_values(by=['연번', '예정시간'])
    print("\n분석 완료.")
    final_result_df
else:
    print("\n분석할 데이터가 없습니다.")

분석 대상 날짜: ['2020-01-06' '2020-01-03' '2020-01-02']


전체 날짜 분석 중: 100%|██████████| 3/3 [00:00<00:00, 14.32it/s]


분석 완료.


## 4. 결과 저장

In [5]:
# 최종 분석 결과를 CSV 파일로 저장합니다.
if 'final_result_df' in locals() and not final_result_df.empty:
    file_path = "./data/bus_818_arrival_analysis.csv"
    final_result_df.to_csv(file_path, index=False)
    print(f"분석 결과가 '{file_path}' 파일로 저장되었습니다.")
else:
    print("저장할 분석 결과가 없습니다.")

분석 결과가 './data/bus_818_arrival_analysis.csv' 파일로 저장되었습니다.
